# Spark: планы выполнения и оптимизация


Читать план выполнения, видеть обмен данными и аккуратно применять вещание и кэш.

Нужны `SparkSessionContext` и таблицы `lakehouse.raw.pg__matches`, `lakehouse.raw.pg__match_players`, `lakehouse.raw.pg__heroes`. Каждая ячейка создаёт отдельное приложение Spark и только читает данные.

Ограничения: Когда нужно объяснить стоимость запроса и найти лишнее чтение или обмен. Вещание допустимо только для небольшого справочника; кэш освобождайте после последнего использования.


## Основные понятия
План строится лениво, а действие запускает вычисление. Обмен данными между узлами обычно возникает при группировке и некоторых объединениях.


In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

## Примеры


**1. Читаемый план**

Постройте план чтения `pg__matches`. В разделе сканирования проверьте список выбранных полей `match_id` и `duration`, а также наличие условия `duration > 1800`.



In [ ]:
with SparkSessionContext('example-plan-read') as spark:
    query = (
        spark
            .table("lakehouse.raw.pg__matches")
            .select("match_id", "duration")
            .filter(F.col("duration") > 1800)
    )
    query.explain(mode="formatted")

**2. План группировки**

Сравните план с предыдущим примером. Операция `Exchange` перед группировкой показывает перераспределение данных между исполнителями Spark.



In [ ]:
with SparkSessionContext('example-plan-group') as spark:
    grouped = (
        spark
            .table("lakehouse.raw.pg__match_players").
            groupBy("hero_id")
            .agg(F.count("*").alias("games"))
            .orderBy(F.desc("games"))
    )
    grouped.explain(mode="formatted")

**3. Соединение со справочником**

Передайте небольшой справочник героев на каждый исполнитель через `broadcast`. В плане должно появиться соединение со справочником, а не перераспределение обеих таблиц.



In [ ]:
with SparkSessionContext('example-plan-broadcast') as spark:
    facts = spark.table("lakehouse.raw.pg__match_players").select("hero_id", "kills")
    heroes = spark.table("lakehouse.raw.pg__heroes").select("hero_id", "localized_name")
    facts.join(F.broadcast(heroes), "hero_id", "left").explain(mode="formatted")

**4. Фильтр и чтение столбцов**

Оставьте два столбца и узкий диапазон длительности. В плане проверьте сокращённый список чтения и условие отбора рядом со сканированием Iceberg.



In [ ]:
with SparkSessionContext('example-plan-filter') as spark:
    (
        spark
            .table("lakehouse.raw.pg__matches")
            .select("match_id", "duration")
            .filter(F.col("duration").between(1800, 3600))
            .limit(5)
            .explain(mode="formatted")
    )


**5. Жизненный цикл кэша**

Материализуйте кэш вызовом `count`, повторно используйте данные для среднего значения и освободите память через `unpersist`.



In [ ]:
with SparkSessionContext('example-plan-cache') as spark:
    frame = spark.table("lakehouse.raw.pg__matches").select("duration").filter("duration > 0").cache()
    print(frame.count())
    
    frame.agg(F.avg("duration")).show()
    frame.unpersist()
